# 01: Fluid Dynamics & LFA Design Optimization

## Overview
This notebook implements a 1D flow rate model based on an extension of Washburn’s Law and Darcy’s Law. The goal is to optimize the geometry of a multi-stage Lateral Flow Assay (LFA) to automate the delivery of nanocatalyst amplification reagents.

### Key Technical Features:
- Parameter Estimation: Uses linear interpolation to derive surface tension and viscosity values for nitrocellulose channels (3 mm, 4 mm) where literature data is unavailable.
- In Silico Design: Simulates "Design 2" of the automated device to determine precise channel lengths (L1, L2, L3) required to meet experimental timelines.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import cumtrapz

# 1. DEFINE PHYSICAL CONSTANTS & INTERPOLATION
# Based on experimental data for 1mm, 2mm, and 5mm channels
widths = [1, 2, 5]
gamma_cos_theta = [25.4, 28.1, 31.2]  # Effective surface tension
viscosity = [0.89, 0.92, 0.98]        # Effective viscosity

def interpolate_params(w_target):
    gamma = np.interp(w_target, widths, gamma_cos_theta)
    mu = np.interp(w_target, widths, viscosity)
    return gamma, mu

# 2. WASHBURN FLOW SIMULATION ENGINE
def simulate_flow(width, max_z=50):
    gamma, mu = interpolate_params(width)
    z = np.linspace(0.1, max_z, 500)  # Distance in mm
    # Simplified Washburn velocity for LFA optimization
    v = (gamma / (2 * mu * z))
    t = cumtrapz(1/v, z, initial=0)   # Time in seconds
    return z, v, t

# 3. OPTIMIZING DESIGN 2 DIMENSIONS
w_main = 3  # Target width for Design 2
z, v, t = simulate_flow(w_main)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(z, v, label=f'{w_main} mm Channel')
plt.title("Velocity vs. Distance")
plt.xlabel("Distance (mm)")
plt.ylabel("Velocity (mm/s)")

plt.subplot(1, 2, 2)
plt.plot(t, z, color='orange')
plt.title("Wicking Distance vs. Time")
plt.xlabel("Time (s)")
plt.ylabel("Distance (mm)")
plt.tight_layout()
plt.show()